In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re


coverage = pd.read_csv("coverage.tsv", sep="\t")

In [2]:
def extract_id(name):
    m = re.search(r"_(\d+sib)_", name)
    return m.group(1) if m else name

def reorder_samples(df):

    meta_cols = ["chr", "start", "genome_pos"]
    sample_cols = [c for c in df.columns if c not in meta_cols]

    name_map = {}
    for c in sample_cols:
        sid = extract_id(c)
        if sid:
            name_map[sid] = c

    def sib_sort(x):
        return int(x.replace("sib", ""))

    ordered_sibs = sorted(name_map.keys(), key=sib_sort)

    ordered_cols = [name_map[s] for s in ordered_sibs]

    return df[meta_cols + ordered_cols], ordered_sibs

In [3]:
def plot_corr_heatmap(df, use="norm", out_file="corr_heatmap_final.png"):

    meta_cols = ["chr", "start", "genome_pos"]
    sample_cols = [c for c in df.columns if c not in meta_cols]

    numeric = df[sample_cols].astype(float)

    if use == "norm":
        auto_mask = ~df["chr"].isin(["chrX","X","chrY","Y"])
        auto_mean = numeric.loc[auto_mask].mean(axis=0)
        data = numeric.div(auto_mean, axis=1)
    else:
        data = numeric

    corr = data.corr()

    new_names = {c: extract_id(c) for c in corr.columns}
    corr.rename(index=new_names, columns=new_names, inplace=True)

    plt.figure(figsize=(10, 8))

    sns.heatmap(
        corr,
        cmap="coolwarm",
        vmin=0,
        vmax=1,
        annot=True,          
        fmt=".2f",           
        square=True,
        linewidths=0.5
    )

    plt.title("Sample correlation")
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.savefig(out_file, dpi=150)
    plt.close()

In [4]:
plot_corr_heatmap(coverage)